# Regresija: od premice do krivulje — B: vaja (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 2, Nadzorovano učenje v praksi**

Predavatelj je v zvezku `A_regresija` pokazal premico in krivulje. Zdaj jih nastavljate sami.
Na vrhu celice zamenjate vrednost in celico ponovno zaženete (**Shift + Enter**).

### Ponovitev
- **Regresija** pomeni, da napovedujemo število. Tu: koliko dni bo trajala dostava naročila.
- Napoved damo **ob oddaji naročila**. Značilke so samo podatki, ki so takrat znani. Podatki, ki nastanejo šele po dostavi, ne smejo med značilke.
- **Časovna delitev:** model učimo na preteklosti in ga preverimo na prihodnosti.
- **Izhodišče:** vedno napovemo isto število (mediano učnih podatkov).
- **MAE** je povprečna napaka v dneh. **RMSE** velike napake šteje bolj.
- **Linearna regresija:** $\hat{y} = b_0 + \beta_1 x$ — presek $b_0$ in naklon $\beta_1$. Učenje je iskanje premice z najmanjšo napako. Značilk je lahko več.
- **Polinomska regresija:** dodamo še stolpce x², x³ … Krivulja se lahko ukrivi. Višja stopnja ni nujno boljša na novih podatkih.

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("olist")
DATA = PODATKI.parents[2]
shema = preberi_shemo(PODATKI / "shema.json")


### Knjižnice in podatki
Funkcije `potence`, `nauci_polinom` in `graf_napovedi` smo napisali v zvezku A; tu jih uvozimo iz `utils/orodja.py`.

In [ ]:
import json
import numpy as np
import pandas as pd
from plotnine import *
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME
from utils.orodja import potence, nauci_polinom, graf_napovedi
nastavi_slog()

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je pripravila skripta `scripts/pripravi_olist.py`. Naredi isto kot zvezka predavatelja
`A2_Priprava_Olist` (mapa `03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov`) in
`A2_Delitev_in_manjkajoce_Olist` (mapa `03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela`).
Tam so vse odločitve z razlogi. Na kratko:

1. Ena vrstica = eno dostavljeno naročilo. Cilj je `dni_dostave`: čas od oddaje naročila do dostave, v dnevih z decimalkami.
2. Napoved damo ob oddaji naročila. Značilke so samo podatki, ki so takrat znani: košarica, izdelki, lokacija, čas naročila in obljubljeni rok. Ocene kupcev in plačila niso značilke.
3. `obljubljeni_dnevi` je rok dostave, ki ga trgovina pokaže kupcu ob nakupu, v dnevih od oddaje naročila.
4. Časovna delitev: učna = kupljeno in dostavljeno pred 1. 3. 2018; validacijska = kupljeno od 1. 3. do 31. 5. 2018 in dostavljeno pred 1. 6. 2018; testna = kupljeno od 1. 6. do 31. 8. 2018.
5. Naročil, ki ob meji še niso bila dostavljena, ni v nobeni množici: njihovega cilja takrat še ne bi poznali. To so predvsem počasne dostave. Validacijska množica ima zato krajše dostave kot vsa naročila iz istega obdobja; testna množica te omejitve nima. Rezultata na validacijski in na testni množici zato nista neposredno primerljiva.
6. Manjkajoče vrednosti so dopolnjene z mediano učne množice.
7. Besedilni stolpci (zvezna država, kategorija) imajo tudi stolpce 0/1, na primer `zvezna_drzava_SP`. Seznam kategorij je iz učne množice.
8. Dolgih dostav ne odstranimo. Zato merimo MAE in RMSE.
9. Značilk vnaprej ne skaliramo. Kjer je to potrebno (potence), to naredimo v zvezku, samo iz učne množice.
10. Testna množica ostane zaprta do tekmovanja na koncu regresijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)
shema = preberi_shemo(PODATKI / "shema.json")   # opis stolpcev
skupine = shema["skupine_za_model"]                                        # skupine stolpcev, pripravljenih za model

X_ucni, y_ucni = ucni[["obljubljeni_dnevi"]], ucni["dni_dostave"]
X_val, y_val = validacijski[["obljubljeni_dnevi"]], validacijski["dni_dostave"]

print(f"učna: {len(ucni)} naročil, validacijska: {len(validacijski)} naročil")

---
## B1. Uganite premico
Nastavite `NAKLON` in `PRESEK` in poskusite dobiti čim manjšo napako **na učnih podatkih**.
Za primerjavo: če vedno napovemo povprečje učnih podatkov (12,9 dneva), je RMSE 9,06 dneva.

Namig: sive pike so povprečja pri vsakem celem obljubljenem dnevu. Kje premica seka navpično os? Za koliko se dvigne na 10 obljubljenih dni?

**Naloga v paru:** primerjajta RMSE. Kdo je bližje?

In [ ]:
NAKLON = 1.0         # ← zamenjajte: za koliko dni se podaljša dostava, če je obljubljeni rok 1 dan daljši
PRESEK = 0.0         # ← zamenjajte: napoved pri 0 obljubljenih dneh

moja_napoved = PRESEK + NAKLON * X_ucni["obljubljeni_dnevi"]
print(f"Vaša premica:  RMSE {root_mean_squared_error(y_ucni, moja_napoved):.2f} dni   MAE {mean_absolute_error(y_ucni, moja_napoved):.2f} dni")
print(f"Izhodišče (povprečje): RMSE {root_mean_squared_error(y_ucni, np.full(len(y_ucni), y_ucni.mean())):.2f} dni")

povprecja = ucni.groupby(ucni["obljubljeni_dnevi"].round())["dni_dostave"].mean().reset_index()     # povprečje pri vsakem celem dnevu
povprecja = povprecja[povprecja["obljubljeni_dnevi"].between(8, 50)]
crta = pd.DataFrame({"obljubljeni_dnevi": [8, 50]})
crta["napoved"] = PRESEK + NAKLON * crta["obljubljeni_dnevi"]

(ggplot(povprecja, aes(x="obljubljeni_dnevi", y="dni_dostave"))
 + geom_point(color=SIVA, size=2)
 + geom_line(aes(y="napoved"), data=crta, color=ORANZNA, size=1.5)
 + coord_cartesian(ylim=(0, 40))
 + labs(title=f"Vaša premica: {PRESEK} + {NAKLON} × obljubljeni_dnevi", x="obljubljeni dnevi", y="dni dostave"))

<details>
<summary><b>Rešitev (klikni)</b></summary>

Linearna regresija najde `presek = 2,02` in `naklon = 0,436`. Njena RMSE na učnih podatkih je 8,36 dneva.

Če uporabimo **isto učno množico in isto značilko** `obljubljeni_dnevi`, nobena druga premica ne more imeti manjše učne RMSE:
`LinearRegression` izbere presek in naklon, ki zmanjšata vsoto kvadratov napak.
Vi ste premico iskali z očmi in poskušanjem, model pa je ta dva parametra izračunal.

</details>

Preverimo: premica, ki jo najde model.

In [ ]:
premica = LinearRegression().fit(X_ucni, y_ucni)
print(f"presek = {premica.intercept_:.2f}, naklon = {premica.coef_[0]:.3f}")
print(f"Učna RMSE: {root_mean_squared_error(y_ucni, premica.predict(X_ucni)):.2f} dni   izhodišče (povprečje {y_ucni.mean():.1f} dni): "
      f"{root_mean_squared_error(y_ucni, np.full(len(y_ucni), y_ucni.mean())):.2f} dni")

## B2. Naloga v paru: mirna in divja krivulja
Imamo samo **40 učnih naročil** (skrajen primer). Oba narišeta krivuljo na istih 40 naročilih. Razlikuje se samo stopnja:
**levi v paru** nastavi `STOPNJA = 2` (mirna krivulja), **desni v paru** `STOPNJA = 10` (divja krivulja).

| | učna MAE (40 naročil) | validacijska MAE |
|---|---|---|
| stopnja 2 | | |
| stopnja 10 | | |

**Vprašanja:** Čigava krivulja gre bliže pikam? Čigava je boljša na novih naročilih? Kaj je podobno kot pri drevesih?

In [ ]:
STOPNJA = 2          # ← stopnja polinoma: levi v paru 2, desni v paru 10

malo = ucni.sample(40, random_state=8)                      # samo 40 učnih naročil
X_malo, y_malo = malo[["obljubljeni_dnevi"]], malo["dni_dostave"]
krivulja = nauci_polinom(X_malo, y_malo, STOPNJA)

# validacijska naročila z obljubljenim rokom znotraj razpona teh 40 naročil
znotraj = validacijski[validacijski["obljubljeni_dnevi"].between(X_malo["obljubljeni_dnevi"].min(), X_malo["obljubljeni_dnevi"].max())]
print(f"Učna MAE (40 naročil): {mean_absolute_error(y_malo, krivulja(X_malo)):.2f} dni")
print(f"Validacijska MAE:      {mean_absolute_error(znotraj['dni_dostave'], krivulja(znotraj[['obljubljeni_dnevi']])):.2f} dni")

(graf_napovedi({f"stopnja {STOPNJA}": krivulja}, malo, povprecja=False)
 + coord_cartesian(ylim=(0, 45))
 + labs(title=f"40 naročil, stopnja {STOPNJA}", x="obljubljeni dnevi", y="dni dostave"))

<details>
<summary><b>Rešitev (klikni)</b></summary>

| | učna MAE (40 naročil) | validacijska MAE |
|---|---|---|
| stopnja 2 | 5,44 | 5,96 |
| stopnja 10 | 4,32 | 11,90 |

- Bliže učnim pikam gre krivulja stopnje 10: njena učna napaka je manjša.
- Na **validacijskih naročilih** je boljša stopnja 2 (MAE 5,96 proti 11,90 dneva). Stopnja 10 v povprečju zgreši približno dvakrat toliko.
- Krivulja stopnje 10 se zvija med posameznimi učnimi pikami. V tem **vzorcu 40 naročil** ni primerov med 39 in 51 obljubljenimi dnevi; tam se krivulja močno odmakne.
- Enako kot pri globokem drevesu: boljše prileganje učnim podatkom, slabša napoved novih. To je pretirano prilagajanje.

</details>

Preverimo obe stopnji naenkrat:

In [ ]:
for st in [2, 10]:
    k = nauci_polinom(X_malo, y_malo, st)
    print(f"stopnja {st:2d}: učna MAE {mean_absolute_error(y_malo, k(X_malo)):.2f}   "
          f"validacijska MAE {mean_absolute_error(znotraj['dni_dostave'], k(znotraj[['obljubljeni_dnevi']])):.2f}")
print("Štirje najdaljši obljubljeni roki v vzorcu:", malo["obljubljeni_dnevi"].nlargest(4).round(1).tolist())

## B3. Najboljša stopnja na vseh podatkih
Zdaj uporabimo vseh 53 644 učnih naročil. Poskusite `STOPNJA = 1, 2, 3, 5, 10, 15`.

**Kako beremo graf:** posameznih naročil na grafu ni; 53 644 pik bi se zlilo v siv oblak.
Vsaka siva pika je **povprečje** vseh učnih naročil v enem od 40 pasov obljubljenega roka.
Model se ne uči iz teh 40 pik, ampak iz vseh 53 644 naročil.

**Vprašanja:** Pri kateri stopnji je validacijska napaka najmanjša? Ali visoka stopnja pomaga?
Zakaj se krivulja stopnje 10 tu ne zvija tako kot v B2?

In [ ]:
STOPNJA = 1          # ← zamenjajte: 1, 2, 3, 5, 10, 15

krivulja = nauci_polinom(X_ucni, y_ucni, STOPNJA)
print(f"Model se uči iz {len(X_ucni)} naročil; sive pike na grafu so njihova povprečja v 40 pasovih.")
print(f"Stopnja {STOPNJA}: učna MAE {mean_absolute_error(y_ucni, krivulja(X_ucni)):.3f}   "
      f"validacijska MAE {mean_absolute_error(y_val, krivulja(X_val)):.3f}   "
      f"validacijska RMSE {root_mean_squared_error(y_val, krivulja(X_val)):.2f}")

graf_napovedi({f"stopnja {STOPNJA}": krivulja}, ucni) + labs(title=f"Polinom stopnje {STOPNJA}", x="obljubljeni dnevi", y="dni dostave")

<details>
<summary><b>Rešitev (klikni)</b></summary>

Stopnja 2 je na **validacijski množici** boljša od premice (MAE 5,51 proti 5,64 dneva; manj je bolje).
Med stopnjami 2 do 5 so razlike majhne (MAE 5,50–5,52); višja stopnja tu ne prinese pomembnega izboljšanja.

Pri stopnjah 10 in 15 je validacijska napaka malo večja (MAE 5,54), učna pa malo manjša (5,175 proti 5,186 pri stopnji 2).
Na tem razrezu podatkov visoka stopnja ne pomaga, pri več kot 50 000 učnih naročilih pa validacijsko napako poslabša le malo.
V vaji B2, kjer smo učili na samo 40 naročilih, je razlika precej večja.

**Zakaj tu ni izrazitega pretiranega prilagajanja kot v B2 in na prosojnici?**

- Primer v B2 in na prosojnici je namenoma skrajen: samo 40 učnih naročil. Polinom stopnje 10 ima 11 števil
  (presek in 10 koeficientov). To je eno število na manj kot 4 naročila, zato krivulja lahko sledi posameznim naročilom.
- Tu se istih 11 števil uči iz 53 644 naročil. Krivulja ne more slediti posameznemu naročilu; opiše lahko samo splošno obliko.
- Pretirano prilagajanje torej ni odvisno samo od stopnje, ampak od tega, kako zapleten je model **glede na količino podatkov**.
  Graf pod zadnjo celico: v tem vzorcu razlika med stopnjama 2 in 10 skoraj izgine že pri 300 učnih naročilih.
- Stopnji 10 in 15 imata enako napako. Visoke potence (x¹⁰, x¹¹ …) so si med seboj skoraj enake,
  zato jih program ne more ločiti in z njimi ne pridobi nič novega.

</details>

Preverimo:

In [ ]:
for st in [1, 2, 3, 4, 5, 10, 15]:
    k = nauci_polinom(X_ucni, y_ucni, st)
    print(f"stopnja {st:2d}: učna MAE {mean_absolute_error(y_ucni, k(X_ucni)):.3f}   validacijska MAE "
          f"{mean_absolute_error(y_val, k(X_val)):.3f}   validacijska RMSE {root_mean_squared_error(y_val, k(X_val)):.2f}")

Preverimo še, zakaj: stopnji 2 in 10 naučimo na vedno več učnih naročilih.
Pri 40 naročilih je to isti vzorec kot v B2. Vzorec je en sam; pri drugem semenu so številke nekoliko drugačne.

In [ ]:
vrstice = []
for n in [40, 100, 300, 1000, 5000, len(ucni)]:
    vzorec = ucni.sample(n, random_state=8)                                  # n naključnih učnih naročil
    X_vzorec, y_vzorec = vzorec[["obljubljeni_dnevi"]], vzorec["dni_dostave"]
    # validacijska naročila z obljubljenim rokom znotraj razpona vzorca (enako kot v B2)
    v_razponu = validacijski[validacijski["obljubljeni_dnevi"].between(X_vzorec["obljubljeni_dnevi"].min(), X_vzorec["obljubljeni_dnevi"].max())]
    mirna, divja = nauci_polinom(X_vzorec, y_vzorec, 2), nauci_polinom(X_vzorec, y_vzorec, 10)
    vrstice.append({"učnih naročil": str(n),
                    "stopnja 2": mean_absolute_error(v_razponu["dni_dostave"], mirna(v_razponu[["obljubljeni_dnevi"]])),
                    "stopnja 10": mean_absolute_error(v_razponu["dni_dostave"], divja(v_razponu[["obljubljeni_dnevi"]]))})
po_velikosti = pd.DataFrame(vrstice)
print("Validacijska MAE (dni):")
print(po_velikosti.round(2).to_string(index=False))

dolga = po_velikosti.melt(id_vars="učnih naročil", var_name="model", value_name="MAE")
dolga["učnih naročil"] = pd.Categorical(dolga["učnih naročil"], po_velikosti["učnih naročil"])     # vrstni red na osi x
dolga["model"] = pd.Categorical(dolga["model"], ["stopnja 2", "stopnja 10"])

(ggplot(dolga, aes(x="učnih naročil", y="MAE", color="model", group="model"))
 + geom_line(size=1.2)
 + geom_point(size=2.5)
 + scale_color_manual(values=[MODRA, ORANZNA])
 + labs(title="Več učnih naročil, manj pretiranega prilagajanja", x="število učnih naročil", y="validacijska MAE (dni)", color=""))

## B4. Naloga v paru: katere značilke pomagajo premici?
Premica ima lahko več značilk. Stolpci so razvrščeni v skupine; besedilni stolpci so v njih že zapisani kot stolpci 0/1.

In [ ]:
pd.DataFrame({"skupina": list(skupine), "stolpcev": [len(s) for s in skupine.values()],
              "prvi trije stolpci": [", ".join(s[:3]) for s in skupine.values()]})

Oba obdržita `obljubljeni_dnevi` in dodata eno skupino stolpcev:
**levi v paru** `skupine["lokacija"]`, **desni v paru** `skupine["cas_narocila"]`.

| | stolpcev | učna MAE | validacijska MAE |
|---|---|---|---|
| samo `obljubljeni_dnevi` | 1 | | |
| + lokacija | | | |
| + cas_narocila | | | |

**Vprašanja:** Katera skupina pomaga na validacijskih naročilih? Pri kateri se učna napaka zmanjša, validacijska pa poveča?

In [ ]:
ZNACILKE = ["obljubljeni_dnevi"] + skupine["lokacija"]      # ← levi v paru: skupine["lokacija"], desni v paru: skupine["cas_narocila"]

ena = LinearRegression().fit(X_ucni, y_ucni)                # za primerjavo: samo obljubljeni_dnevi
vec = LinearRegression().fit(ucni[ZNACILKE], y_ucni)
print(f"samo obljubljeni_dnevi: učna MAE {mean_absolute_error(y_ucni, ena.predict(X_ucni)):.2f}   "
      f"validacijska MAE {mean_absolute_error(y_val, ena.predict(X_val)):.2f}")
print(f"{len(ZNACILKE)} stolpcev:            učna MAE {mean_absolute_error(y_ucni, vec.predict(ucni[ZNACILKE])):.2f}   "
      f"validacijska MAE {mean_absolute_error(y_val, vec.predict(validacijski[ZNACILKE])):.2f}")

<details>
<summary><b>Rešitev (klikni)</b></summary>

| | stolpcev | učna MAE | validacijska MAE |
|---|---|---|---|
| samo `obljubljeni_dnevi` | 1 | 5,24 | 5,64 |
| + lokacija | 40 | 4,95 | 5,20 |
| + cas_narocila | 13 | 5,15 | 6,18 |

- Lokacija pomaga: obe napaki se zmanjšata. Kje sta kupec in prodajalec, veliko pove o času dostave.
- Z značilkami časa naročila se učna MAE zmanjša, validacijska pa **poveča** (s 5,64 na 6,18 dneva).
  Učna naročila so iz let 2016 do februarja 2018, validacijska pa iz marca do maja 2018.
  Časovni premik je možna razlaga; iz teh dveh številk samega vzroka ne moremo dokazati.
- Več stolpcev torej ni vedno bolje. Odloča validacijska napaka, ne učna.

</details>

Preverimo obe skupini in obdobji obeh množic:

In [ ]:
for ime in ["lokacija", "cas_narocila"]:
    stolpci = ["obljubljeni_dnevi"] + skupine[ime]
    m = LinearRegression().fit(ucni[stolpci], y_ucni)
    print(f"+ {ime:13s} {len(stolpci):3d} stolpcev: učna MAE {mean_absolute_error(y_ucni, m.predict(ucni[stolpci])):.2f}   "
          f"validacijska MAE {mean_absolute_error(y_val, m.predict(validacijski[stolpci])):.2f}")
print("Nakupi, učna množica:        ", ucni["datum_nakupa"].min().date(), "do", ucni["datum_nakupa"].max().date())
print("Nakupi, validacijska množica:", validacijski["datum_nakupa"].min().date(), "do", validacijski["datum_nakupa"].max().date())

---
## B5. Moj kandidat za tekmovanje
Sestavite svoj najboljši model. Izberite stolpce (`ZNACILKE`) in stopnjo (`STOPNJA`): 1 je premica, 2 ali 3 je polinom.
Stolpce lahko naštejete po imenih ali seštejete skupine, na primer `skupine["obljuba"] + skupine["lokacija"] + ["postnina"]`.

Zapišite si izpisani slovar. Testno množico odpremo samo v tekmovalnem zvezku, na koncu regresijskega dela.

In [ ]:
ZNACILKE = ["obljubljeni_dnevi", "postnina"]      # ← vaš izbor stolpcev
STOPNJA = 1                                       # ← 1 = premica, 2 ali 3 = polinom

napovej = nauci_polinom(ucni[ZNACILKE], y_ucni, STOPNJA)
napoved = napovej(validacijski[ZNACILKE])
print(f"Validacijska MAE {mean_absolute_error(y_val, napoved):.3f} dni   RMSE {root_mean_squared_error(y_val, napoved):.3f} dni   "
      f"(izhodišče: MAE {mean_absolute_error(y_val, np.full(len(y_val), y_ucni.median())):.3f})")

# "seme" je None: linearna in polinomska regresija nimata naključja; ključ ostane, da imajo vsi kandidati enake ključe
kandidat = {"model": "linearna" if STOPNJA == 1 else "polinomska", "znacilke": ZNACILKE,
            "nastavitve": {"stopnja": STOPNJA}, "seme": None, "mae_validacija": round(mean_absolute_error(y_val, napoved), 3)}
kandidat